## Concept 4 — SummaryBuffer Memory

### What is it?
Best of both worlds: `trim_messages` keeps recent messages **raw**, and the trimmed-out older
messages are compressed into a **summary** via LLM. Both are injected into the prompt.

### Pipeline
```
All messages: [H1 A1 H2 A2 H3 A3 H4 A4 H5 A5]
  ↓
trim_messages (keep last 3 turns):
  Trimmed out : [H1 A1 H2 A2]  → LLM summarizes → 1 SystemMessage
  Kept raw    :         [H3 A3 H4 A4 H5 A5]
  ↓
Prompt = [SystemMessage(summary)] + [H3 A3 H4 A4 H5 A5] + [new question]
```

### Why This Is the Best Practical Strategy
```
Buffer       : exact recall, grows forever
Window       : bounded, forgets old facts
Summary      : compact, lossy, extra LLM cost
SummaryBuffer: bounded + old facts preserved in summary  ← best balance
```

In [ ]:
print('hi')

In [ ]:
import sys, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, '.')

from MemoryUtils import get_llm, TEST_TURNS, make_history, load_test_history, run_recall_test
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.messages import trim_messages, SystemMessage, HumanMessage, AIMessage

llm = get_llm()
print('LLM ready')

### Step 1 — Split Messages: Trimmed vs Kept

In [ ]:
history = make_history()
load_test_history(history)

K = 3  # keep last 3 turns raw

kept = trim_messages(
    history.messages,
    max_tokens=K * 2,
    token_counter=lambda msgs: len(msgs),
    strategy='last',
    start_on='human',
)

# Messages that were trimmed out = everything not in 'kept'
kept_set    = set(id(m) for m in kept)
trimmed_out = [m for m in history.messages if id(m) not in kept_set]

print(f'Total messages : {len(history.messages)}')
print(f'Trimmed out    : {len(trimmed_out)} ({len(trimmed_out)//2} turns) → will be summarized')
print(f'Kept raw       : {len(kept)} ({len(kept)//2} turns) → injected as-is')

### Step 2 — Summarize the Trimmed Messages

In [ ]:
def summarize_messages(messages: list) -> str:
    """Compress a list of messages into a brief summary string."""
    if not messages:
        return ''
    lines = []
    for m in messages:
        role = 'Human' if isinstance(m, HumanMessage) else 'AI'
        lines.append(f'{role}: {m.content}')
    conversation = '\n'.join(lines)
    return llm.invoke(f'Summarize this conversation in 2-3 sentences:\n{conversation}').content

old_summary = summarize_messages(trimmed_out)
print('Summary of trimmed messages:')
print(old_summary)

### Step 3 — Inspect the Final Prompt Structure
Show what the LLM receives: 1 summary SystemMessage + raw recent turns.

In [ ]:
final_messages = [SystemMessage(content=f'Summary of earlier conversation: {old_summary}')] + kept

print('What the LLM sees in the prompt:')
print('=' * 55)
for msg in final_messages:
    if isinstance(msg, SystemMessage):
        print(f'[SUMMARY]  {msg.content[:120]}...' if len(msg.content) > 120 else f'[SUMMARY]  {msg.content}')
    elif isinstance(msg, HumanMessage):
        print(f'[Human]    {msg.content}')
    else:
        print(f'[AI]       {msg.content[:80]}...' if len(msg.content) > 80 else f'[AI]       {msg.content}')
    print()

### Full Function — chat_with_summary_buffer

In [ ]:
K = 3

chat_history = make_history()
load_test_history(chat_history)

def get_summary_buffer_messages(messages: list, k: int) -> list:
    """Return [SystemMessage(summary of old)] + [last k turns raw]."""
    kept = trim_messages(
        messages,
        max_tokens=k * 2,
        token_counter=lambda msgs: len(msgs),
        strategy='last',
        start_on='human',
    )
    kept_ids    = set(id(m) for m in kept)
    trimmed_out = [m for m in messages if id(m) not in kept_ids]

    if not trimmed_out:
        return kept  # nothing to summarize

    summary = summarize_messages(trimmed_out)
    return [SystemMessage(content=f'Earlier conversation: {summary}')] + kept

def chat_with_summary_buffer(query):
    context_messages = get_summary_buffer_messages(chat_history.messages, K)
    context_messages.append(HumanMessage(content=query))
    response = llm.invoke(context_messages)
    chat_history.add_user_message(query)
    chat_history.add_ai_message(response.content)
    return response.content

### Test — Recall via SummaryBuffer
Recent turns are raw + old turns summarized. All 5 facts should be recalled.

In [ ]:
run_recall_test(chat_with_summary_buffer, label='Concept 4 -- SummaryBuffer Memory')